# ASL-Tutor: Model Training Notebook

This notebook trains a CNN to recognize American Sign Language (ASL) alphabet signs.

## Setup
1. Download the ASL Alphabet dataset from Kaggle
2. Extract to `data/asl_alphabet/` with subfolders `asl_alphabet_train/` and `asl_alphabet_test/`
3. Run this notebook to train the model

In [ ]:
# Install dependencies (if needed)
# !pip install torch torchvision numpy pandas matplotlib tqdm pillow onnx

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Check device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. Dataset Loading & Preprocessing

In [ ]:
from src.dataset import get_dataloaders, ASL_CLASSES, denormalize

# Configuration
DATA_ROOT = 'data/asl_alphabet'
BATCH_SIZE = 64
VAL_SPLIT = 0.1

# Get dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    num_workers=4
)

In [ ]:
# Visualize sample images
def show_batch(loader, n_images=8):
    images, labels = next(iter(loader))
    
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < n_images:
            img = denormalize(images[i]).permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            ax.set_title(f"Label: {ASL_CLASSES[labels[i]]}")
            ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Sample training images:")
show_batch(train_loader)

## 2. Model Architecture

In [ ]:
from src.model import ASLClassifier, count_parameters

# Create model
model = ASLClassifier(num_classes=len(ASL_CLASSES), pretrained=True, dropout=0.2)
model = model.to(device)

print(f"Model: ASLClassifier (MobileNetV3-Small backbone)")
print(f"Total parameters: {count_parameters(model):,}")
print(f"Output classes: {len(ASL_CLASSES)}")

In [ ]:
# Test forward pass
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

## 3. Training

In [ ]:
from src.train import Trainer

# Training configuration
EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 5

# Create trainer
trainer = Trainer(
    model=model,
    device=device,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING
)

print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")

In [ ]:
# Train the model
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    save_dir='models'
)

In [ ]:
# Plot learning curves
trainer.plot_learning_curves(save_path='models/learning_curves.png')

## 4. Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
model.load_state_dict(torch.load('models/asl_cnn_best.pt', map_location=device))
model.eval()

# Evaluate on validation set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=ASL_CLASSES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ASL_CLASSES, yticklabels=ASL_CLASSES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=150)
plt.show()

## 5. Export to ONNX

In [ ]:
from src.model import export_to_onnx

# Export model to ONNX
export_to_onnx(
    model=model,
    output_path='models/asl_cnn.onnx',
    device='cpu'
)

## 6. Inference Test

In [ ]:
from src.inference import ASLPredictor

# Create predictor
predictor = ASLPredictor(model_path='models/asl_cnn_best.pt')

# Test on validation images
images, labels = next(iter(val_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < 8:
        # Denormalize for display
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        
        # Predict
        img_array = (img * 255).astype(np.uint8)
        pred_label, confidence = predictor.predict_sign(img_array)
        true_label = ASL_CLASSES[labels[i]]
        
        # Display
        color = 'green' if pred_label == true_label else 'red'
        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                    color=color, fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('models/inference_test.png', dpi=150)
plt.show()

## 7. Summary

The model has been trained and saved:
- PyTorch weights: `models/asl_cnn_best.pt`
- ONNX model: `models/asl_cnn.onnx`

You can now:
1. Run the webcam demo: `python -m src.inference`
2. Start the API server: `python -m src.api`
3. Open the frontend: `frontend/index.html`